# HOG Features + Random Forest — Image Classification Pipeline

**Feature descriptor:** Histogram of Oriented Gradients (HOG) &nbsp;|&nbsp; **Classifier:** Random Forest Classifier &nbsp;|&nbsp; **Validation:** 5-fold cross-validation

---

This notebook implements an end-to-end image-classification experiment:

1. **Feature Extraction** — the Histogram of Oriented Gradients descriptor (9 orientation bins, 8x8-pixel cells, 2x2-cell blocks, L2-Hys normalization) is computed for each image.
2. **Data Loading** — images are read fold-by-fold from a directory structure of `train` / `test` splits.
3. **Model Training & Evaluation** — a **Random Forest** — `RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)` is trained on each fold, across six image resolutions, and evaluated with accuracy, precision, recall, and F1 (weighted & macro).
4. **Results** — per-fold metrics, 5-fold averages, and confusion matrices are reported; results are exported to CSV.

**Outputs**

| File | Contents |
|---|---|
| `HOG_RandomForest_5Fold_All_Results.csv` | Metrics for every fold x image size |
| `HOG_RandomForest_5Fold_Average_Results.csv` | Metrics averaged over the 5 folds |


## 1&nbsp;&nbsp;Imports & Logging

Standard scientific-Python stack: OpenCV and scikit-image for image processing, scikit-learn for the classifier and metrics, and matplotlib/seaborn for visualization. Logging is configured to report progress and any per-image failures without halting the experiment.


In [ ]:
import os
import logging
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import seaborn as sns

from tqdm import tqdm
from joblib import Parallel, delayed

from skimage import io, color
from skimage.transform import resize
from skimage.feature import hog

from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix
)


# ======================================================
# LOGGING
# ======================================================

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)




## 2&nbsp;&nbsp;HOG Feature Extraction

`extract_hog_features(image, target_size)` converts the image to grayscale, resizes it to `target_size`, then computes the Histogram of Oriented Gradients descriptor:

| Parameter | Value |
|---|---|
| Orientation bins | 9 |
| Pixels per cell | 8 x 8 |
| Cells per block | 2 x 2 |
| Block normalization | L2-Hys |

The block-normalized gradient histograms are returned as a single flattened feature vector. Extraction failures are logged and the image is skipped.


In [ ]:
# ======================================================
# HOG FEATURE EXTRACTION
# ======================================================

def extract_hog_features(image, target_size):
    try:
        # Convert RGB to grayscale if needed
        if image.ndim == 3:
            gray_image = color.rgb2gray(image)
        else:
            gray_image = image.astype(np.float32) / 255.0

        # Resize image
        resized_image = resize(
            gray_image,
            target_size,
            anti_aliasing=True
        )

        # HOG feature extraction
        features = hog(
            resized_image,
            orientations=9,
            pixels_per_cell=(8, 8),
            cells_per_block=(2, 2),
            block_norm="L2-Hys",
            visualize=False,
            feature_vector=True
        )

        return np.array(features, dtype=np.float32)

    except Exception as e:
        logging.error(f"HOG feature extraction failed: {e}")
        return None




## 3&nbsp;&nbsp;Data Loading

`load_data(directory, target_size)` discovers class sub-directories, extracts HOG features for every image (classes processed in parallel across all CPU cores), and returns the feature matrix `X`, integer-encoded labels `y`, and the ordered list of class names.


In [ ]:
# ======================================================
# DATA LOADING FUNCTION
# ======================================================

def load_data(directory, target_size):

    class_names = sorted([
        d
        for d in os.listdir(directory)
        if os.path.isdir(os.path.join(directory, d))
    ])

    def process_class(class_name):

        class_path = os.path.join(directory, class_name)

        feature_list = []
        label_list = []

        for file_name in tqdm(
            os.listdir(class_path),
            desc=f"Processing {class_name}"
        ):

            # Skip hidden files
            if file_name.startswith("."):
                continue

            image_path = os.path.join(class_path, file_name)

            try:
                image = io.imread(image_path)

            except Exception as e:
                logging.warning(f"Cannot read {image_path}: {e}")
                continue

            features = extract_hog_features(image, target_size)

            if features is not None:
                feature_list.append(features)
                label_list.append(class_name)

        return feature_list, label_list

    # Parallel feature extraction
    results = Parallel(n_jobs=-1)(
        delayed(process_class)(cls)
        for cls in tqdm(class_names, desc="Loading Classes")
    )

    X = []
    y = []

    for features, labels in results:
        X.extend(features)
        y.extend(labels)

    # Convert labels to numeric values
    y = np.array([
        class_names.index(label)
        for label in y
    ])

    return np.array(X), y, class_names




## 4&nbsp;&nbsp;Experiment Configuration

Defines the dataset location, the five cross-validation folds, and the six image resolutions to be evaluated. `all_results` accumulates the metrics from every (image size, fold) run.


In [ ]:
# ======================================================
# 5-FOLD SETTINGS
# ======================================================

baseDir = r"E:\THUSHAR\DATASET\Croped_5Fold"


# For Google Colab use:
# baseDir = "/content/drive/MyDrive/Croped_5Fold"


folds = [
    "fold_1",
    "fold_2",
    "fold_3",
    "fold_4",
    "fold_5"
]


# ======================================================
# DIFFERENT IMAGE SIZES
# ======================================================

sizes = [
    (8, 8),
    (16, 16),
    (32, 32),
    (64, 64),
    (128, 128),
    (196, 210)
]


# ======================================================
# STORE ALL RESULTS
# ======================================================

all_results = []




## 5&nbsp;&nbsp;Training & Evaluation — Main Experiment Loop

For every image size and every fold:

1. Load the fold's train and test sets.
2. Train a **Random Forest** — `RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)`.
3. Predict on the test set and compute **accuracy**, **precision / recall / F1** (weighted and macro).
4. Append the metrics to `all_results` and plot the **confusion matrix**.


In [ ]:
# ======================================================
# MAIN EXPERIMENT LOOP
# ======================================================

for size in sizes:

    print("\n========================================")
    print(f"PROCESSING IMAGE SIZE: {size[0]}x{size[1]}")
    print("========================================")

    for fold in folds:

        print(f"\n########## {fold} ##########")

        mainDir = os.path.join(baseDir, fold)

        # --------------------------------------------------
        # LOAD DATA
        # --------------------------------------------------

        X_train, y_train, class_names = load_data(
            os.path.join(mainDir, "train"),
            size
        )

        X_test, y_test, _ = load_data(
            os.path.join(mainDir, "test"),
            size
        )

        print("Train shape:", X_train.shape)
        print("Test shape :", X_test.shape)

        # --------------------------------------------------
        # RANDOM FOREST CLASSIFIER
        # --------------------------------------------------

        clf = RandomForestClassifier(
            n_estimators=100,
            random_state=42,
            n_jobs=-1
        )

        # --------------------------------------------------
        # TRAIN
        # --------------------------------------------------

        print("\nTraining Random Forest ...")

        clf.fit(X_train, y_train)

        # --------------------------------------------------
        # PREDICT
        # --------------------------------------------------

        y_pred = clf.predict(X_test)

        # ==================================================
        # METRICS
        # ==================================================

        accuracy = accuracy_score(y_test, y_pred)

        precision_w, recall_w, f1_w, _ = precision_recall_fscore_support(
            y_test,
            y_pred,
            average="weighted",
            zero_division=0
        )

        precision_m, recall_m, f1_m, _ = precision_recall_fscore_support(
            y_test,
            y_pred,
            average="macro",
            zero_division=0
        )

        # ==================================================
        # DISPLAY METRICS
        # ==================================================

        print(f"Accuracy            : {accuracy:.4f}")
        print(f"F1 Weighted         : {f1_w:.4f}")
        print(f"F1 Macro            : {f1_m:.4f}")

        # ==================================================
        # SAVE RESULTS
        # ==================================================

        all_results.append({
            "Fold": fold,
            "Feature": "HOG",
            "Classifier": "RandomForest",
            "Image_Size": f"{size[0]}x{size[1]}",
            "Accuracy": accuracy,
            "Precision_Weighted": precision_w,
            "Recall_Weighted": recall_w,
            "F1_Weighted": f1_w,
            "Precision_Macro": precision_m,
            "Recall_Macro": recall_m,
            "F1_Macro": f1_m
        })

        # ==================================================
        # CONFUSION MATRIX
        # ==================================================

        cm = confusion_matrix(y_test, y_pred)

        plt.figure(figsize=(10, 8))

        sns.heatmap(
            cm,
            annot=True,
            fmt="d",
            cmap="Blues",
            xticklabels=class_names,
            yticklabels=class_names
        )

        plt.title(
            f"HOG + Random Forest\n"
            f"{fold} ({size[0]}x{size[1]})"
        )

        plt.xlabel("Predicted Class")
        plt.ylabel("Actual Class")
        plt.xticks(rotation=90)
        plt.yticks(rotation=0)
        plt.tight_layout()
        plt.show()




## 6&nbsp;&nbsp;Results Aggregation & Export

Fold-wise metrics are saved to CSV, averaged across the five folds per image size, exported, and displayed sorted by image size and accuracy.


In [ ]:
# ======================================================
# SAVE FOLD-WISE RESULTS
# ======================================================

results_df = pd.DataFrame(all_results)

results_df.to_csv(
    "HOG_RandomForest_5Fold_All_Results.csv",
    index=False
)

print(
    "\nFold-wise results saved: "
    "HOG_RandomForest_5Fold_All_Results.csv"
)


# ======================================================
# COMPUTE AVERAGE 5-FOLD RESULTS
# ======================================================

average_df = results_df.groupby(
    [
        "Feature",
        "Classifier",
        "Image_Size"
    ]
).agg({
    "Accuracy": "mean",
    "Precision_Weighted": "mean",
    "Recall_Weighted": "mean",
    "F1_Weighted": "mean",
    "Precision_Macro": "mean",
    "Recall_Macro": "mean",
    "F1_Macro": "mean"
}).reset_index()


# ======================================================
# SAVE AVERAGE RESULTS
# ======================================================

average_df.to_csv(
    "HOG_RandomForest_5Fold_Average_Results.csv",
    index=False
)

print(
    "Average results saved: "
    "HOG_RandomForest_5Fold_Average_Results.csv"
)


# ======================================================
# DISPLAY FINAL RESULTS
# ======================================================

print("\n========================================")
print("FINAL 5-FOLD AVERAGE RESULTS")
print("========================================")

print(
    average_df.sort_values(
        ["Image_Size", "Accuracy"],
        ascending=[True, False]
    )
)


print("\n========================================")
print("RANDOM FOREST EXPERIMENTS COMPLETED SUCCESSFULLY")
print("========================================")